In [ ]:
using Pkg
Pkg.activate("ift-env")
Pkg.add(["IceFloeTracker", "Images", "DataFrames", "CSV", "CairoMakie"])

In [ ]:
using IceFloeTracker, Images, DataFrames, CSV, CairoMakie

# Case 006 Baffin Bay
This image pair comes from the [Ice Floe Validation Dataset](github.org/danielmwatkins/ice_floe_validation_dataset). The scene is 100 km by 100 km. Some floes are only partially visible, but many can be seen in both images. The images are near enough in time that true matches will overlap each other in space.

In [ ]:
fnames = [
    "006-baffin_bay-100km-20220530.terra.truecolor.250m.tiff",
    "006-baffin_bay-100km-20220530.aqua.truecolor.250m.tiff"
    ]

# Add passtimes from data table

# Load images and cast to grayscale
tc_imgs = load.(joinpath.("../data/images/", fnames))
gray_imgs = tc_imgs .|> r -> Gray.(r)
# Adjust image contrast
gray_imgs .= adjust_histogram.(gray_imgs, [ContrastLimitedAdaptiveHistogramEqualization(; nbins=255, rblocks=4, cblocks=4, clip=2)])
binary_imgs = binarize.(gray_imgs, [AdaptiveThreshold(; window_size=200, percentage=0)]);

In [ ]:
# Visualize: Top is true color, middle has enhanced contrast, bottom is binarized
mosaicview(tc_imgs..., gray_imgs..., binary_imgs..., nrow=3, rowmajor=true, npad=10, fillvalue=RGBA(0,0,0,0))

In [ ]:
vsr(r) = RGBA.(view_seg_random(SegmentedImage(r .> 0, r)))
labeled_imgs = dist_morph_split.(binary_imgs .|> r -> r .> 0; max_depth=10, opening_strel=strel_box((3,3)))
mosaicview(vsr.(labeled_imgs)..., nrow=1, npad=10, fillvalue=RGBA(0, 0, 0, 0))

In [ ]:
floe_tracker = IceFloeTracker.FSPipeline.Track();
tracked_floes = floe_tracker.(

# Fram Strait / Lopez-Acosta et al. 2019
This sequence of images was used in a test case for the original Ice Floe Tracker paper.